# Feature Correlation & Redundancy Analysis for BGP Anomaly Detection

This notebook performs comprehensive **intra-view and cross-view redundancy analysis** across all
BGP feature sets:

1. **Traditional BGP features** (25 features from `feature_exctration.ipynb`): announcements, withdrawals,
   edit distances, AS path metrics, origin attributes, implicit withdrawals, rare ASes, flaps, NADAS
2. **Graph-level structural features** (from `bgp_graph_features.ipynb`): diameter, density, clustering
   coefficient, centrality statistics, connected components, etc.
3. **Graph-level relationship features**: provider/customer/peer edge fractions, valley-free violations, etc.
4. **Node-level structural features**: degree centrality, betweenness, PageRank, etc.
5. **Node-level relationship features**: provider/customer/peer counts, ratios, etc.

## Methods Applied

| # | Method | What it captures | Output |
|---|--------|-----------------|--------|
| 1 | **Distance Correlation (dCor)** | Non-linear + linear dependencies | Pairwise matrix |
| 2 | **Spearman Rank Correlation** | Monotonic relationships | Pairwise matrix |
| 3 | **Pearson Correlation** | Linear relationships | Pairwise matrix |
| 4 | **Variance Inflation Factor (VIF)** | Multicollinearity | Per-feature score |
| 5 | **Hierarchical Clustering** | Collinear feature groups | Dendrogram + clusters |
| 6 | **mRMR (Min Redundancy Max Relevance)** | Relevance vs. redundancy trade-off | Ranked feature list |
| 7 | **Mutual Information** | General statistical dependence | Pairwise matrix |
| 8 | **Condition Number Analysis** | Numerical stability / collinearity | Scalar per view |
| 9 | **Cross-View Correlation** | Inter-view dependencies | Heatmaps between views |

Each intra-view method is applied **separately to each feature view**, followed by a
**cross-view analysis** that examines correlations between traditional and graph features.

## 1. Installation & Imports

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install dcor scikit-learn scipy numpy pandas matplotlib seaborn statsmodels mrmr-selection

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

# Correlation methods
from scipy import stats as sp_stats
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
import dcor  # Distance correlation

# Multicollinearity
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

# Mutual Information
from sklearn.feature_selection import mutual_info_regression

# mRMR
try:
    from mrmr import mrmr_regression
    HAS_MRMR = True
except ImportError:
    HAS_MRMR = False
    print("WARNING: mrmr-selection not installed. Install with: pip install mrmr-selection")

warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams.update({
    'figure.figsize': (14, 10),
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

print("All imports successful.")
print(f"  dcor version: {dcor.__version__}")
print(f"  mRMR available: {HAS_MRMR}")

## 2. Configuration & Data Loading

In [ ]:
# ============================================================================
# CONFIGURATION - Point these to your extracted feature CSVs
# ============================================================================

# Base directory where bgp_graph_features.ipynb wrote its output
BASE_DIR = Path("/home/smotaali/First_Full_Paper/bgp_graph_features_results")
OUTPUT_DIR = BASE_DIR / "output"

# Output directory for this analysis
ANALYSIS_DIR = BASE_DIR / "redundancy_analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

# File naming pattern from bgp_graph_features.ipynb
COLLECTOR = "rrc04"
TARGET_AS = 12880
EGO_K_HOP = 2
START_DATE = "2025-11-07"
END_DATE = "2025-11-08"

suffix = f"{COLLECTOR}_AS{TARGET_AS}_{EGO_K_HOP}hop_{START_DATE}_{END_DATE}"

GRAPH_TS_PATH = OUTPUT_DIR / f"graph_level_timeseries_{suffix}.csv"
NODE_TS_PATH = OUTPUT_DIR / f"node_level_timeseries_{suffix}.csv"

# Traditional BGP features CSV (from feature_exctration.ipynb)
TRADITIONAL_FEATURES_PATH = Path("/home/smotaali/BGP_Traffic_Generation/results/20251208_223239_exctracted_1s.csv")

# Redundancy threshold for correlation-based removal
CORRELATION_THRESHOLD = 0.90  # |r| > this -> consider redundant
VIF_THRESHOLD = 10.0          # VIF > this -> multicollinear
CLUSTER_DISTANCE_THRESHOLD = 0.3  # 1 - |r| distance for hierarchical clustering

print(f"Graph-level CSV:         {GRAPH_TS_PATH}")
print(f"Node-level CSV:          {NODE_TS_PATH}")
print(f"Traditional features CSV: {TRADITIONAL_FEATURES_PATH}")
print(f"Analysis output:         {ANALYSIS_DIR}")

In [ ]:
# ============================================================================
# Load Data
# ============================================================================

graph_ts_df = pd.read_csv(GRAPH_TS_PATH)
node_ts_df = pd.read_csv(NODE_TS_PATH)
traditional_df = pd.read_csv(TRADITIONAL_FEATURES_PATH)

print(f"Graph-level time series:    {graph_ts_df.shape}")
print(f"Node-level time series:     {node_ts_df.shape}")
print(f"  Unique snapshots: {node_ts_df['snapshot_id'].nunique()}")
print(f"  Unique ASes: {node_ts_df['asn'].nunique()}")
print(f"Traditional BGP features:   {traditional_df.shape}")
print(f"  Columns: {list(traditional_df.columns)}")

In [ ]:
# ============================================================================
# Separate feature views
# ============================================================================

META_COLS = ['snapshot_id', 'timestamp', 'collector']
NODE_META_COLS = ['snapshot_id', 'timestamp', 'collector', 'asn']

# Boolean/flag columns to exclude from numeric analysis
FLAG_COLS = ['diameter_approximate', 'symmetry_ratio_partial', 'natural_connectivity_partial']

# --- Traditional BGP feature columns ---
TRADITIONAL_META_COLS = ['window_start', 'window_end', 'label']
traditional_feature_cols = [c for c in traditional_df.columns
                            if c not in TRADITIONAL_META_COLS]

# --- Graph-level features ---
graph_feature_cols = [c for c in graph_ts_df.columns
                      if c not in META_COLS and c not in FLAG_COLS]

# Split into structural vs. relationship features
RELATIONSHIP_GRAPH_FEATURES = [
    'rel_coverage', 'frac_p2c_edges', 'frac_p2p_edges', 'frac_unknown_edges',
    'n_tier1_in_subgraph', 'n_ixp_in_subgraph', 'valley_free_violations',
    'valley_free_violation_rate', 'valley_free_paths_checked',
    'avg_violation_depth', 'max_violation_depth'
]

RELATIONSHIP_NODE_FEATURES = [
    'n_providers', 'n_customers', 'n_peers', 'n_unknown_rel',
    'p2c_ratio', 'p2p_ratio', 'provider_ratio', 'is_tier1', 'is_ixp'
]

# Structural graph-level (the original 16 features + derived)
structural_graph_cols = [c for c in graph_feature_cols
                         if c not in RELATIONSHIP_GRAPH_FEATURES
                         and c in graph_ts_df.columns]

# Relationship graph-level
rel_graph_cols = [c for c in RELATIONSHIP_GRAPH_FEATURES
                  if c in graph_ts_df.columns]

# Node-level structural features
node_all_cols = [c for c in node_ts_df.columns if c not in NODE_META_COLS]
structural_node_cols = [c for c in node_all_cols if c not in RELATIONSHIP_NODE_FEATURES]
rel_node_cols = [c for c in RELATIONSHIP_NODE_FEATURES if c in node_ts_df.columns]

print("Feature Views:")
print(f"  0. Traditional BGP: {len(traditional_feature_cols)} features")
print(f"     {traditional_feature_cols}")
print(f"  1. Structural graph-level: {len(structural_graph_cols)} features")
print(f"     {structural_graph_cols}")
print(f"  2. Relationship graph-level: {len(rel_graph_cols)} features")
print(f"     {rel_graph_cols}")
print(f"  3. Structural node-level: {len(structural_node_cols)} features")
print(f"     {structural_node_cols}")
print(f"  4. Relationship node-level: {len(rel_node_cols)} features")
print(f"     {rel_node_cols}")

In [ ]:
# ============================================================================
# Prepare clean numeric DataFrames for each view
# ============================================================================

def prepare_view(df, feature_cols, view_name):
    """
    Extract numeric features, drop constant/all-NaN columns, impute remaining NaNs.
    Returns cleaned DataFrame and list of dropped columns.
    """
    available = [c for c in feature_cols if c in df.columns]
    view_df = df[available].copy()

    # Convert to numeric, coerce errors
    view_df = view_df.apply(pd.to_numeric, errors='coerce')

    # Drop columns that are all NaN
    all_nan = view_df.columns[view_df.isna().all()].tolist()
    if all_nan:
        print(f"  [{view_name}] Dropping all-NaN columns: {all_nan}")
        view_df = view_df.drop(columns=all_nan)

    # Drop constant columns (zero variance)
    constant = view_df.columns[view_df.nunique() <= 1].tolist()
    if constant:
        print(f"  [{view_name}] Dropping constant columns: {constant}")
        view_df = view_df.drop(columns=constant)

    # Report NaN summary
    nan_pct = view_df.isna().mean()
    cols_with_nan = nan_pct[nan_pct > 0]
    if len(cols_with_nan) > 0:
        print(f"  [{view_name}] Columns with NaN (will be median-imputed):")
        for col, pct in cols_with_nan.items():
            print(f"    {col}: {pct:.1%} missing")

    # Impute NaNs with column median
    view_df = view_df.fillna(view_df.median())

    dropped = all_nan + constant
    print(f"  [{view_name}] Final shape: {view_df.shape} "
          f"(dropped {len(dropped)} columns)")
    return view_df, dropped


# Prepare all views
print("=" * 60)
print("Preparing feature views...")
print("=" * 60)

# Traditional BGP features
traditional_clean_df, traditional_dropped = prepare_view(
    traditional_df, traditional_feature_cols, "Traditional BGP")

# For graph-level: each row is a snapshot (time series)
struct_graph_df, struct_graph_dropped = prepare_view(
    graph_ts_df, structural_graph_cols, "Structural Graph")

rel_graph_df, rel_graph_dropped = prepare_view(
    graph_ts_df, rel_graph_cols, "Relationship Graph")

# For node-level: aggregate across snapshots (mean per ASN) to get
# a single feature matrix for correlation analysis
node_agg = node_ts_df.groupby('asn')[node_all_cols].mean()

struct_node_df, struct_node_dropped = prepare_view(
    node_agg, structural_node_cols, "Structural Node")

rel_node_df, rel_node_dropped = prepare_view(
    node_agg, rel_node_cols, "Relationship Node")

# Collect all views in a dict for systematic processing
VIEWS = {
    'Traditional BGP': traditional_clean_df,
    'Structural Graph-Level': struct_graph_df,
    'Relationship Graph-Level': rel_graph_df,
    'Structural Node-Level': struct_node_df,
    'Relationship Node-Level': rel_node_df,
}

print("\n" + "=" * 60)
print("Summary of Views")
print("=" * 60)
for name, df in VIEWS.items():
    print(f"  {name}: {df.shape[0]} samples x {df.shape[1]} features")

---
## 3. Distance Correlation (dCor)

Distance correlation (Székely et al., 2007) captures **both linear and non-linear** dependencies.
- dCor = 0 iff the variables are **independent** (stronger than Pearson = 0)
- dCor ∈ [0, 1]
- Particularly useful for detecting non-linear redundancies that Pearson/Spearman miss

In [ ]:
def compute_distance_correlation_matrix(df, view_name):
    """
    Compute pairwise distance correlation matrix.
    This is O(n^2 * p^2) so can be slow for large datasets.
    For node-level with many rows, we subsample.
    """
    MAX_SAMPLES = 2000  # subsample for speed
    if len(df) > MAX_SAMPLES:
        print(f"  [{view_name}] Subsampling {MAX_SAMPLES}/{len(df)} rows for dCor")
        df_sub = df.sample(n=MAX_SAMPLES, random_state=42)
    else:
        df_sub = df

    cols = df_sub.columns.tolist()
    n_features = len(cols)
    dcor_matrix = np.ones((n_features, n_features))

    for i in range(n_features):
        for j in range(i + 1, n_features):
            x = df_sub[cols[i]].values
            y = df_sub[cols[j]].values
            dc = dcor.distance_correlation(x, y)
            dcor_matrix[i, j] = dc
            dcor_matrix[j, i] = dc

    dcor_df = pd.DataFrame(dcor_matrix, index=cols, columns=cols)
    return dcor_df


print("Computing Distance Correlation matrices...")
dcor_results = {}
for name, df in VIEWS.items():
    if df.shape[1] < 2:
        print(f"  [{name}] Skipping (< 2 features)")
        continue
    print(f"  [{name}] {df.shape[1]} features...")
    dcor_results[name] = compute_distance_correlation_matrix(df, name)
    print(f"    Done.")

print("\nDistance correlation computation complete.")

In [ ]:
# Visualize dCor matrices
for name, dcor_df in dcor_results.items():
    fig, ax = plt.subplots(figsize=(max(10, dcor_df.shape[1] * 0.6),
                                    max(8, dcor_df.shape[1] * 0.5)))
    mask = np.triu(np.ones_like(dcor_df, dtype=bool), k=1)
    sns.heatmap(dcor_df, mask=mask, annot=dcor_df.shape[1] <= 20,
                fmt='.2f', cmap='YlOrRd', vmin=0, vmax=1,
                square=True, linewidths=0.5, ax=ax,
                annot_kws={'size': 8})
    ax.set_title(f'Distance Correlation — {name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig(ANALYSIS_DIR / f'dcor_{name.replace(" ", "_").lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # Report highly correlated pairs
    high_dcor = []
    for i in range(dcor_df.shape[0]):
        for j in range(i + 1, dcor_df.shape[1]):
            if dcor_df.iloc[i, j] >= CORRELATION_THRESHOLD:
                high_dcor.append((
                    dcor_df.index[i], dcor_df.columns[j],
                    dcor_df.iloc[i, j]
                ))
    if high_dcor:
        print(f"\n[{name}] Highly distance-correlated pairs (dCor >= {CORRELATION_THRESHOLD}):")
        for f1, f2, val in sorted(high_dcor, key=lambda x: -x[2]):
            print(f"  {f1} <-> {f2}: dCor = {val:.4f}")
    else:
        print(f"\n[{name}] No pairs exceed dCor threshold {CORRELATION_THRESHOLD}")

---
## 4. Spearman & Pearson Correlation

- **Pearson**: measures linear correlation (assumes normality)
- **Spearman**: measures monotonic relationships (rank-based, robust to outliers)

Both are fast to compute and widely used as baseline redundancy detectors.

In [ ]:
def compute_correlations(df, view_name):
    """
    Compute Pearson and Spearman correlation matrices with p-values.
    """
    cols = df.columns.tolist()
    n = len(cols)

    # Pearson
    pearson_r = df.corr(method='pearson')
    pearson_p = pd.DataFrame(np.ones((n, n)), index=cols, columns=cols)
    for i in range(n):
        for j in range(i + 1, n):
            r, p = sp_stats.pearsonr(df[cols[i]], df[cols[j]])
            pearson_p.iloc[i, j] = p
            pearson_p.iloc[j, i] = p

    # Spearman
    spearman_r = df.corr(method='spearman')
    spearman_p = pd.DataFrame(np.ones((n, n)), index=cols, columns=cols)
    for i in range(n):
        for j in range(i + 1, n):
            rho, p = sp_stats.spearmanr(df[cols[i]], df[cols[j]])
            spearman_p.iloc[i, j] = p
            spearman_p.iloc[j, i] = p

    return {
        'pearson_r': pearson_r, 'pearson_p': pearson_p,
        'spearman_r': spearman_r, 'spearman_p': spearman_p
    }


print("Computing Pearson & Spearman correlations...")
corr_results = {}
for name, df in VIEWS.items():
    if df.shape[1] < 2:
        print(f"  [{name}] Skipping (< 2 features)")
        continue
    print(f"  [{name}]...")
    corr_results[name] = compute_correlations(df, name)

print("Done.")

In [ ]:
# Visualize Pearson & Spearman side by side
for name, res in corr_results.items():
    fig, axes = plt.subplots(1, 2, figsize=(
        max(20, res['pearson_r'].shape[1] * 1.0),
        max(8, res['pearson_r'].shape[1] * 0.5)))

    for ax, method, matrix in zip(axes,
                                   ['Pearson', 'Spearman'],
                                   [res['pearson_r'], res['spearman_r']]):
        mask = np.triu(np.ones_like(matrix, dtype=bool), k=1)
        sns.heatmap(matrix, mask=mask, annot=matrix.shape[1] <= 20,
                    fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                    square=True, linewidths=0.5, ax=ax,
                    annot_kws={'size': 7})
        ax.set_title(f'{method} — {name}', fontsize=13, fontweight='bold')

    plt.tight_layout()
    fig.savefig(ANALYSIS_DIR / f'pearson_spearman_{name.replace(" ", "_").lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # Report redundant pairs
    for method, r_matrix in [('Pearson', res['pearson_r']),
                              ('Spearman', res['spearman_r'])]:
        redundant = []
        for i in range(r_matrix.shape[0]):
            for j in range(i + 1, r_matrix.shape[1]):
                if abs(r_matrix.iloc[i, j]) >= CORRELATION_THRESHOLD:
                    redundant.append((
                        r_matrix.index[i], r_matrix.columns[j],
                        r_matrix.iloc[i, j]
                    ))
        if redundant:
            print(f"\n[{name}] {method} redundant pairs (|r| >= {CORRELATION_THRESHOLD}):")
            for f1, f2, val in sorted(redundant, key=lambda x: -abs(x[2])):
                print(f"  {f1} <-> {f2}: r = {val:.4f}")
        else:
            print(f"\n[{name}] {method}: no pairs exceed |r| >= {CORRELATION_THRESHOLD}")

---
## 5. Variance Inflation Factor (VIF) — Multicollinearity

VIF measures how much the variance of a regression coefficient is inflated due to collinearity.
- VIF = 1: no collinearity
- VIF = 5–10: moderate collinearity
- VIF > 10: severe multicollinearity (candidate for removal)

Reference: Belsley, Kuh & Welsch (1980). *Regression Diagnostics*.

In [ ]:
def compute_vif(df, view_name):
    """
    Compute VIF for each feature. Features are standardized first.
    For node-level with many rows, subsample for speed.
    """
    MAX_SAMPLES = 5000
    if len(df) > MAX_SAMPLES:
        print(f"  [{view_name}] Subsampling {MAX_SAMPLES}/{len(df)} rows for VIF")
        df_sub = df.sample(n=MAX_SAMPLES, random_state=42)
    else:
        df_sub = df

    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_sub)
    X_df = pd.DataFrame(X_scaled, columns=df_sub.columns)

    vif_data = []
    for i, col in enumerate(X_df.columns):
        try:
            vif_val = variance_inflation_factor(X_df.values, i)
        except Exception:
            vif_val = np.inf
        vif_data.append({'Feature': col, 'VIF': vif_val})

    vif_df = pd.DataFrame(vif_data).sort_values('VIF', ascending=False)
    return vif_df


print("Computing VIF for each view...")
print("=" * 60)
vif_results = {}
for name, df in VIEWS.items():
    if df.shape[1] < 2:
        print(f"  [{name}] Skipping (< 2 features)")
        continue
    print(f"\n[{name}]")
    vif_df = compute_vif(df, name)
    vif_results[name] = vif_df
    print(vif_df.to_string(index=False))

    multicollinear = vif_df[vif_df['VIF'] > VIF_THRESHOLD]
    if len(multicollinear) > 0:
        print(f"  >>> {len(multicollinear)} features with VIF > {VIF_THRESHOLD} "
              f"(multicollinear)")
    else:
        print(f"  >>> No features with VIF > {VIF_THRESHOLD}")

In [ ]:
# Visualize VIF as bar charts
for name, vif_df in vif_results.items():
    # Cap infinite VIF for visualization
    vif_plot = vif_df.copy()
    vif_plot['VIF'] = vif_plot['VIF'].replace([np.inf], vif_plot['VIF']
                                              [vif_plot['VIF'] != np.inf].max() * 1.5
                                              if (vif_plot['VIF'] != np.inf).any()
                                              else 100)

    fig, ax = plt.subplots(figsize=(max(10, len(vif_plot) * 0.5), 6))
    colors = ['#d32f2f' if v > VIF_THRESHOLD else '#1976d2'
              for v in vif_plot['VIF']]
    bars = ax.barh(vif_plot['Feature'], vif_plot['VIF'], color=colors)
    ax.axvline(x=VIF_THRESHOLD, color='red', linestyle='--', linewidth=1.5,
               label=f'VIF = {VIF_THRESHOLD}')
    ax.axvline(x=5, color='orange', linestyle='--', linewidth=1,
               label='VIF = 5 (moderate)')
    ax.set_xlabel('Variance Inflation Factor')
    ax.set_title(f'VIF — {name}', fontsize=14, fontweight='bold')
    ax.legend()
    ax.invert_yaxis()
    plt.tight_layout()
    fig.savefig(ANALYSIS_DIR / f'vif_{name.replace(" ", "_").lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

---
## 6. Hierarchical Clustering on Feature Correlations

Cluster features based on correlation distance `d = 1 - |corr|`.
Features within the same cluster (below a distance threshold) are near-duplicates;
we can keep one representative per cluster.

Reference: Hastie, Tibshirani & Friedman (2009). *Elements of Statistical Learning*.

In [ ]:
def hierarchical_cluster_features(df, view_name, threshold=None):
    """
    Perform hierarchical clustering on features using 1 - |Spearman| as distance.
    Returns cluster assignments and the linkage matrix.
    """
    if threshold is None:
        threshold = CLUSTER_DISTANCE_THRESHOLD

    corr = df.corr(method='spearman').abs()
    # Convert to distance: d = 1 - |r|
    dist_matrix = 1 - corr
    np.fill_diagonal(dist_matrix.values, 0)

    # Ensure symmetry and valid condensed form
    dist_matrix = (dist_matrix + dist_matrix.T) / 2
    condensed = squareform(dist_matrix.values, checks=False)

    # Ward-like linkage on correlation distance
    Z = linkage(condensed, method='average')
    clusters = fcluster(Z, t=threshold, criterion='distance')

    cluster_df = pd.DataFrame({
        'Feature': df.columns,
        'Cluster': clusters
    }).sort_values('Cluster')

    return Z, cluster_df, dist_matrix


print("Hierarchical Clustering of Features")
print("=" * 60)
cluster_results = {}

for name, df in VIEWS.items():
    if df.shape[1] < 3:
        print(f"  [{name}] Skipping (< 3 features)")
        continue

    Z, cluster_df, dist_matrix = hierarchical_cluster_features(df, name)
    cluster_results[name] = {'linkage': Z, 'clusters': cluster_df,
                              'dist_matrix': dist_matrix}

    print(f"\n[{name}]")
    n_clusters = cluster_df['Cluster'].nunique()
    print(f"  {df.shape[1]} features -> {n_clusters} clusters "
          f"(threshold = {CLUSTER_DISTANCE_THRESHOLD})")

    for cid in sorted(cluster_df['Cluster'].unique()):
        members = cluster_df[cluster_df['Cluster'] == cid]['Feature'].tolist()
        if len(members) > 1:
            print(f"  Cluster {cid} ({len(members)} features — REDUNDANT GROUP): {members}")
        else:
            print(f"  Cluster {cid}: {members[0]}")

In [ ]:
# Dendrograms
for name, res in cluster_results.items():
    Z = res['linkage']
    cluster_df = res['clusters']
    labels = VIEWS[name].columns.tolist()

    fig, ax = plt.subplots(figsize=(max(12, len(labels) * 0.6), 8))
    dn = dendrogram(Z, labels=labels, ax=ax, leaf_rotation=90,
                    color_threshold=CLUSTER_DISTANCE_THRESHOLD,
                    above_threshold_color='gray')
    ax.axhline(y=CLUSTER_DISTANCE_THRESHOLD, color='red', linestyle='--',
               linewidth=1.5, label=f'Threshold = {CLUSTER_DISTANCE_THRESHOLD}')
    ax.set_ylabel('Distance (1 - |Spearman|)')
    ax.set_title(f'Feature Dendrogram — {name}', fontsize=14, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    fig.savefig(ANALYSIS_DIR / f'dendrogram_{name.replace(" ", "_").lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

---
## 7. mRMR (Minimum Redundancy Maximum Relevance)

mRMR selects features that are **maximally relevant** to a target variable while being
**minimally redundant** among themselves.

Since this is an **unsupervised** intra-view analysis, we use a proxy target:
- For **graph-level**: use the first principal component (PC1) as a synthetic target
- For **node-level**: use degree centrality as the target (a natural proxy for node importance)

Reference: Peng, Long & Ding (2005). *IEEE Trans. PAMI*.

In [ ]:
from sklearn.decomposition import PCA


def run_mrmr(df, view_name, target_col=None, n_select=None):
    """
    Run mRMR feature selection.

    If target_col is None, uses PC1 as a synthetic target.
    Returns ranked list of selected features.
    """
    if not HAS_MRMR:
        print(f"  [{view_name}] mRMR not available. Skipping.")
        return None

    if n_select is None:
        n_select = max(3, df.shape[1] // 2)

    if target_col and target_col in df.columns:
        y = df[target_col]
        X = df.drop(columns=[target_col])
        print(f"  [{view_name}] Using '{target_col}' as target")
    else:
        # Use PC1 as synthetic target
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(df)
        pca = PCA(n_components=1, random_state=42)
        y = pd.Series(pca.fit_transform(X_scaled).ravel(),
                      index=df.index, name='PC1')
        X = df
        explained = pca.explained_variance_ratio_[0]
        print(f"  [{view_name}] Using PC1 as target "
              f"(explains {explained:.1%} variance)")

    n_select = min(n_select, X.shape[1])

    selected = mrmr_regression(
        X=X, y=y, K=n_select, relevance='f', redundancy='c',
        show_progress=False
    )
    return selected


print("mRMR Feature Selection")
print("=" * 60)
mrmr_results = {}

for name, df in VIEWS.items():
    if df.shape[1] < 3:
        print(f"  [{name}] Skipping (< 3 features)")
        continue

    # Choose target based on view type
    if 'Node' in name:
        target = 'degree_centrality' if 'degree_centrality' in df.columns else None
    else:
        target = None  # use PC1

    selected = run_mrmr(df, name, target_col=target)
    mrmr_results[name] = selected

    if selected:
        print(f"  [{name}] mRMR ranking ({len(selected)} features):")
        for rank, feat in enumerate(selected, 1):
            print(f"    {rank}. {feat}")
        eliminated = [f for f in df.columns if f not in selected
                      and f != 'degree_centrality']
        if eliminated:
            print(f"  Eliminated by mRMR: {eliminated}")
    print()

---
## 8. Mutual Information (MI)

Mutual information measures **general statistical dependence** (not just linear or monotonic).
Unlike dCor, MI is based on density estimation and can capture more complex dependencies.

Reference: Kraskov, Stögbauer & Grassberger (2004). *Physical Review E*.

In [ ]:
def compute_mutual_information_matrix(df, view_name):
    """
    Compute pairwise mutual information matrix using sklearn's KNN-based estimator.
    Normalized to [0, 1] using MI(X,Y) / sqrt(MI(X,X) * MI(Y,Y)).
    """
    MAX_SAMPLES = 3000
    if len(df) > MAX_SAMPLES:
        print(f"  [{view_name}] Subsampling {MAX_SAMPLES}/{len(df)} rows for MI")
        df_sub = df.sample(n=MAX_SAMPLES, random_state=42)
    else:
        df_sub = df

    cols = df_sub.columns.tolist()
    n = len(cols)

    # Compute self-MI for normalization
    self_mi = {}
    for col in cols:
        x = df_sub[col].values.reshape(-1, 1)
        mi = mutual_info_regression(x, df_sub[col].values,
                                     n_neighbors=5, random_state=42)
        self_mi[col] = mi[0]

    # Compute pairwise MI
    mi_matrix = np.zeros((n, n))
    np.fill_diagonal(mi_matrix, 1.0)

    for i in range(n):
        X_others = df_sub.drop(columns=[cols[i]]).values
        mi_scores = mutual_info_regression(
            X_others, df_sub[cols[i]].values,
            n_neighbors=5, random_state=42
        )
        other_cols = [c for c in cols if c != cols[i]]
        for j_idx, other_col in enumerate(other_cols):
            j = cols.index(other_col)
            raw_mi = mi_scores[j_idx]
            # Normalized MI
            denom = np.sqrt(self_mi[cols[i]] * self_mi[other_col])
            nmi = raw_mi / denom if denom > 0 else 0
            mi_matrix[i, j] = min(nmi, 1.0)  # clip to [0, 1]

    # Symmetrize
    mi_matrix = (mi_matrix + mi_matrix.T) / 2
    np.fill_diagonal(mi_matrix, 1.0)

    return pd.DataFrame(mi_matrix, index=cols, columns=cols)


print("Computing Mutual Information matrices...")
mi_results = {}
for name, df in VIEWS.items():
    if df.shape[1] < 2:
        print(f"  [{name}] Skipping (< 2 features)")
        continue
    print(f"  [{name}]...")
    mi_results[name] = compute_mutual_information_matrix(df, name)
    print(f"    Done.")

print("\nMutual information computation complete.")

In [ ]:
# Visualize MI matrices
for name, mi_df in mi_results.items():
    fig, ax = plt.subplots(figsize=(max(10, mi_df.shape[1] * 0.6),
                                    max(8, mi_df.shape[1] * 0.5)))
    mask = np.triu(np.ones_like(mi_df, dtype=bool), k=1)
    sns.heatmap(mi_df, mask=mask, annot=mi_df.shape[1] <= 20,
                fmt='.2f', cmap='YlGnBu', vmin=0, vmax=1,
                square=True, linewidths=0.5, ax=ax,
                annot_kws={'size': 8})
    ax.set_title(f'Normalized Mutual Information — {name}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig(ANALYSIS_DIR / f'mi_{name.replace(" ", "_").lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # Report high-MI pairs
    high_mi = []
    for i in range(mi_df.shape[0]):
        for j in range(i + 1, mi_df.shape[1]):
            if mi_df.iloc[i, j] >= CORRELATION_THRESHOLD:
                high_mi.append((
                    mi_df.index[i], mi_df.columns[j], mi_df.iloc[i, j]
                ))
    if high_mi:
        print(f"\n[{name}] High NMI pairs (>= {CORRELATION_THRESHOLD}):")
        for f1, f2, val in sorted(high_mi, key=lambda x: -x[2]):
            print(f"  {f1} <-> {f2}: NMI = {val:.4f}")
    else:
        print(f"\n[{name}] No pairs exceed NMI threshold {CORRELATION_THRESHOLD}")

---
## 9. Condition Number Analysis

The condition number of the feature correlation matrix indicates overall numerical
stability and multicollinearity:
- CN < 30: well-conditioned
- CN 30–100: moderate collinearity
- CN > 100: severe collinearity
- CN > 1000: near-singular (extreme multicollinearity)

Reference: Belsley, Kuh & Welsch (1980). *Regression Diagnostics*.

In [ ]:
print("Condition Number Analysis")
print("=" * 60)

for name, df in VIEWS.items():
    if df.shape[1] < 2:
        continue

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df)
    corr_matrix = np.corrcoef(X_scaled.T)

    eigenvalues = np.linalg.eigvalsh(corr_matrix)
    eigenvalues = np.sort(eigenvalues)[::-1]

    # Condition number = sqrt(max_eig / min_eig)
    min_eig = eigenvalues[eigenvalues > 1e-12].min() if (eigenvalues > 1e-12).any() else 1e-12
    condition_number = np.sqrt(eigenvalues[0] / min_eig)

    if condition_number < 30:
        status = "WELL-CONDITIONED"
    elif condition_number < 100:
        status = "MODERATE COLLINEARITY"
    elif condition_number < 1000:
        status = "SEVERE COLLINEARITY"
    else:
        status = "NEAR-SINGULAR (EXTREME)"

    print(f"\n[{name}]")
    print(f"  Condition number: {condition_number:.1f} -> {status}")
    print(f"  Eigenvalue spectrum: max={eigenvalues[0]:.4f}, "
          f"min={min_eig:.6f}")
    print(f"  Near-zero eigenvalues (< 0.01): "
          f"{np.sum(eigenvalues < 0.01)}")

    # Plot eigenvalue spectrum
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(range(1, len(eigenvalues) + 1), eigenvalues, color='steelblue')
    ax.axhline(y=0.01, color='red', linestyle='--', label='Near-zero threshold')
    ax.set_xlabel('Eigenvalue Index')
    ax.set_ylabel('Eigenvalue')
    ax.set_title(f'Correlation Matrix Eigenvalues — {name}\n'
                 f'Condition Number = {condition_number:.1f} ({status})',
                 fontsize=12, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    fig.savefig(ANALYSIS_DIR / f'condition_{name.replace(" ", "_").lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ============================================================================
# Cross-View Correlation: Traditional BGP vs Graph-Level Features
# ============================================================================

def cross_view_correlation(df_a, df_b, name_a, name_b, method='spearman'):
    """
    Compute cross-view correlation matrix between two feature DataFrames.
    Rows are aligned by index (assumes same ordering/alignment).
    Returns correlation matrix of shape (features_a x features_b).
    """
    # Use the minimum number of samples from both views
    n = min(len(df_a), len(df_b))
    a = df_a.iloc[:n].reset_index(drop=True)
    b = df_b.iloc[:n].reset_index(drop=True)

    cross_corr = pd.DataFrame(
        np.zeros((a.shape[1], b.shape[1])),
        index=a.columns, columns=b.columns
    )
    cross_pval = cross_corr.copy()

    for i, col_a in enumerate(a.columns):
        for j, col_b in enumerate(b.columns):
            if method == 'spearman':
                r, p = sp_stats.spearmanr(a[col_a], b[col_b])
            else:
                r, p = sp_stats.pearsonr(a[col_a], b[col_b])
            cross_corr.iloc[i, j] = r
            cross_pval.iloc[i, j] = p

    return cross_corr, cross_pval


print("=" * 80)
print("CROSS-VIEW CORRELATION ANALYSIS")
print("=" * 80)

# Define view pairs for cross-view analysis
cross_view_pairs = []

# Traditional vs each graph view
for graph_name in ['Structural Graph-Level', 'Relationship Graph-Level']:
    if graph_name in VIEWS and VIEWS[graph_name].shape[1] >= 2:
        cross_view_pairs.append(('Traditional BGP', graph_name))

# Also compare structural vs relationship graph features
if ('Structural Graph-Level' in VIEWS and 'Relationship Graph-Level' in VIEWS
    and VIEWS['Structural Graph-Level'].shape[1] >= 2
    and VIEWS['Relationship Graph-Level'].shape[1] >= 2):
    cross_view_pairs.append(('Structural Graph-Level', 'Relationship Graph-Level'))

cross_view_results = {}

for name_a, name_b in cross_view_pairs:
    print(f"\n--- {name_a} vs {name_b} ---")

    for method in ['spearman', 'pearson']:
        cross_corr, cross_pval = cross_view_correlation(
            VIEWS[name_a], VIEWS[name_b], name_a, name_b, method=method
        )

        key = f"{name_a}_vs_{name_b}_{method}"
        cross_view_results[key] = {
            'corr': cross_corr, 'pval': cross_pval,
            'name_a': name_a, 'name_b': name_b, 'method': method
        }

        # Visualize
        fig, ax = plt.subplots(figsize=(
            max(10, cross_corr.shape[1] * 0.8),
            max(6, cross_corr.shape[0] * 0.5)))
        sns.heatmap(cross_corr, annot=cross_corr.shape[1] <= 15,
                    fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                    linewidths=0.5, ax=ax, annot_kws={'size': 7})
        ax.set_title(f'Cross-View {method.title()} Correlation\n{name_a} vs {name_b}',
                     fontsize=13, fontweight='bold')
        ax.set_ylabel(name_a)
        ax.set_xlabel(name_b)
        plt.tight_layout()
        fname = f'cross_view_{method}_{name_a.replace(" ", "_").lower()}_vs_{name_b.replace(" ", "_").lower()}.png'
        fig.savefig(ANALYSIS_DIR / fname, dpi=150, bbox_inches='tight')
        plt.show()

        # Report strong cross-view correlations
        strong = []
        for i in range(cross_corr.shape[0]):
            for j in range(cross_corr.shape[1]):
                if abs(cross_corr.iloc[i, j]) >= 0.7:  # Lower threshold for cross-view
                    strong.append((
                        cross_corr.index[i], cross_corr.columns[j],
                        cross_corr.iloc[i, j], cross_pval.iloc[i, j]
                    ))

        if strong:
            print(f"  [{method.title()}] Strong cross-view pairs (|r| >= 0.7):")
            for f1, f2, r, p in sorted(strong, key=lambda x: -abs(x[2])):
                sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
                print(f"    {f1} <-> {f2}: r = {r:.4f} (p = {p:.2e}) {sig}")
        else:
            print(f"  [{method.title()}] No strong cross-view pairs (|r| >= 0.7)")

# Summary statistics
print("\n" + "=" * 60)
print("Cross-View Summary: Mean |correlation| between views")
print("=" * 60)
for key, res in cross_view_results.items():
    if 'spearman' in key:
        mean_abs = res['corr'].abs().values.mean()
        max_abs = res['corr'].abs().values.max()
        print(f"  {res['name_a']} vs {res['name_b']}:")
        print(f"    Mean |Spearman|: {mean_abs:.4f}, Max |Spearman|: {max_abs:.4f}")
        if mean_abs < 0.3:
            print(f"    -> Views are largely COMPLEMENTARY (low shared information)")
        elif mean_abs < 0.6:
            print(f"    -> Views have MODERATE shared information")
        else:
            print(f"    -> Views are highly REDUNDANT (consider feature selection across views)")

---
## 10. Cross-View Correlation Analysis

Examines correlations **between** feature views to understand how traditional BGP features
relate to graph-level features. This is critical for multi-view learning:
- High cross-view correlation means views share redundant information
- Low cross-view correlation means views capture complementary aspects

We align views by timestamp where possible, or compare aggregate statistics.

---
## 11. Consolidated Redundancy Report & Feature Selection

Combine evidence from all methods to make final recommendations about which features to keep/remove.

In [ ]:
def build_redundancy_report(view_name, df, corr_res, dcor_res, vif_res,
                             cluster_res, mrmr_res, mi_res):
    """
    Build a consolidated per-feature redundancy score based on all methods.
    Higher score = more evidence of redundancy.
    """
    features = df.columns.tolist()
    report = pd.DataFrame({'Feature': features})

    # --- 1. Max |Pearson| with any other feature ---
    if corr_res and 'pearson_r' in corr_res:
        pearson = corr_res['pearson_r'].abs()
        np.fill_diagonal(pearson.values, 0)
        report['Max_Pearson'] = [pearson.loc[f].max() for f in features]
    else:
        report['Max_Pearson'] = np.nan

    # --- 2. Max |Spearman| with any other feature ---
    if corr_res and 'spearman_r' in corr_res:
        spearman = corr_res['spearman_r'].abs()
        np.fill_diagonal(spearman.values, 0)
        report['Max_Spearman'] = [spearman.loc[f].max() for f in features]
    else:
        report['Max_Spearman'] = np.nan

    # --- 3. Max dCor with any other feature ---
    if dcor_res is not None:
        dcor_abs = dcor_res.copy()
        np.fill_diagonal(dcor_abs.values, 0)
        report['Max_dCor'] = [dcor_abs.loc[f].max() for f in features]
    else:
        report['Max_dCor'] = np.nan

    # --- 4. Max NMI with any other feature ---
    if mi_res is not None:
        mi_abs = mi_res.copy()
        np.fill_diagonal(mi_abs.values, 0)
        report['Max_NMI'] = [mi_abs.loc[f].max() for f in features]
    else:
        report['Max_NMI'] = np.nan

    # --- 5. VIF ---
    if vif_res is not None:
        vif_map = dict(zip(vif_res['Feature'], vif_res['VIF']))
        report['VIF'] = report['Feature'].map(vif_map)
    else:
        report['VIF'] = np.nan

    # --- 6. Cluster size (features in multi-member clusters are redundant) ---
    if cluster_res:
        cluster_df = cluster_res['clusters']
        cluster_sizes = cluster_df.groupby('Cluster')['Feature'].transform('count')
        cluster_map = dict(zip(cluster_df['Feature'], cluster_sizes))
        report['Cluster_Size'] = report['Feature'].map(cluster_map).fillna(1)
    else:
        report['Cluster_Size'] = 1

    # --- 7. mRMR rank (not selected = penalized) ---
    if mrmr_res:
        mrmr_rank = {f: rank for rank, f in enumerate(mrmr_res, 1)}
        report['mRMR_Rank'] = report['Feature'].map(mrmr_rank)
        report['mRMR_Selected'] = report['mRMR_Rank'].notna()
    else:
        report['mRMR_Rank'] = np.nan
        report['mRMR_Selected'] = np.nan

    # --- Composite redundancy score ---
    # Count how many methods flag each feature as redundant
    report['N_Flags'] = 0
    if 'Max_Pearson' in report:
        report['N_Flags'] += (report['Max_Pearson'] >= CORRELATION_THRESHOLD).astype(int)
    if 'Max_Spearman' in report:
        report['N_Flags'] += (report['Max_Spearman'] >= CORRELATION_THRESHOLD).astype(int)
    if 'Max_dCor' in report:
        report['N_Flags'] += (report['Max_dCor'] >= CORRELATION_THRESHOLD).astype(int)
    if 'Max_NMI' in report:
        report['N_Flags'] += (report['Max_NMI'] >= CORRELATION_THRESHOLD).astype(int)
    if 'VIF' in report:
        report['N_Flags'] += (report['VIF'] > VIF_THRESHOLD).astype(int)
    report['N_Flags'] += (report['Cluster_Size'] > 1).astype(int)
    if 'mRMR_Selected' in report and report['mRMR_Selected'].notna().any():
        report['N_Flags'] += (~report['mRMR_Selected'].fillna(True)).astype(int)

    report = report.sort_values('N_Flags', ascending=False)

    return report


print("\n" + "=" * 80)
print("CONSOLIDATED REDUNDANCY REPORT")
print("=" * 80)

redundancy_reports = {}
for name, df in VIEWS.items():
    if df.shape[1] < 2:
        continue

    report = build_redundancy_report(
        view_name=name,
        df=df,
        corr_res=corr_results.get(name),
        dcor_res=dcor_results.get(name),
        vif_res=vif_results.get(name),
        cluster_res=cluster_results.get(name),
        mrmr_res=mrmr_results.get(name),
        mi_res=mi_results.get(name)
    )
    redundancy_reports[name] = report

    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")

    # Format for display
    display_cols = ['Feature', 'Max_Pearson', 'Max_Spearman', 'Max_dCor',
                    'Max_NMI', 'VIF', 'Cluster_Size', 'mRMR_Rank', 'N_Flags']
    display_cols = [c for c in display_cols if c in report.columns]
    print(report[display_cols].to_string(index=False, float_format='%.3f'))

    # Recommendations
    highly_redundant = report[report['N_Flags'] >= 4]['Feature'].tolist()
    moderately_redundant = report[
        (report['N_Flags'] >= 2) & (report['N_Flags'] < 4)
    ]['Feature'].tolist()
    clean = report[report['N_Flags'] < 2]['Feature'].tolist()

    print(f"\nRecommendations:")
    if highly_redundant:
        print(f"  REMOVE (flagged by 4+ methods): {highly_redundant}")
    if moderately_redundant:
        print(f"  REVIEW (flagged by 2-3 methods): {moderately_redundant}")
    print(f"  KEEP (flagged by 0-1 methods):   {clean}")

In [ ]:
# Visualize the consolidated report as a heatmap
for name, report in redundancy_reports.items():
    # Select numeric score columns
    score_cols = ['Max_Pearson', 'Max_Spearman', 'Max_dCor', 'Max_NMI']
    score_cols = [c for c in score_cols if c in report.columns
                  and report[c].notna().any()]

    if not score_cols:
        continue

    plot_df = report.set_index('Feature')[score_cols].copy()
    plot_df = plot_df.sort_index()

    fig, ax = plt.subplots(figsize=(10, max(6, len(plot_df) * 0.4)))
    sns.heatmap(plot_df, annot=True, fmt='.2f', cmap='YlOrRd',
                vmin=0, vmax=1, linewidths=0.5, ax=ax,
                annot_kws={'size': 9})
    ax.set_title(f'Max Pairwise Dependency per Feature — {name}',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Method')
    plt.tight_layout()
    fig.savefig(ANALYSIS_DIR / f'summary_{name.replace(" ", "_").lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

---
## 12. Export Results

In [ ]:
# ============================================================================
# Export all results
# ============================================================================

# 1. Redundancy reports
for name, report in redundancy_reports.items():
    fname = f'redundancy_report_{name.replace(" ", "_").lower()}.csv'
    report.to_csv(ANALYSIS_DIR / fname, index=False)
    print(f"Saved: {ANALYSIS_DIR / fname}")

# 2. Correlation matrices
for name, res in corr_results.items():
    base = name.replace(" ", "_").lower()
    res['pearson_r'].to_csv(ANALYSIS_DIR / f'pearson_{base}.csv')
    res['spearman_r'].to_csv(ANALYSIS_DIR / f'spearman_{base}.csv')

# 3. Distance correlation matrices
for name, dcor_df in dcor_results.items():
    base = name.replace(" ", "_").lower()
    dcor_df.to_csv(ANALYSIS_DIR / f'dcor_{base}.csv')

# 4. MI matrices
for name, mi_df in mi_results.items():
    base = name.replace(" ", "_").lower()
    mi_df.to_csv(ANALYSIS_DIR / f'mi_{base}.csv')

# 5. VIF results
for name, vif_df in vif_results.items():
    base = name.replace(" ", "_").lower()
    vif_df.to_csv(ANALYSIS_DIR / f'vif_{base}.csv', index=False)

# 6. Cluster assignments
for name, res in cluster_results.items():
    base = name.replace(" ", "_").lower()
    res['clusters'].to_csv(ANALYSIS_DIR / f'clusters_{base}.csv', index=False)

# 7. mRMR rankings
for name, selected in mrmr_results.items():
    if selected:
        base = name.replace(" ", "_").lower()
        mrmr_df = pd.DataFrame({
            'Rank': range(1, len(selected) + 1),
            'Feature': selected
        })
        mrmr_df.to_csv(ANALYSIS_DIR / f'mrmr_{base}.csv', index=False)

# 8. Cross-view correlation matrices
for key, res in cross_view_results.items():
    fname = f'cross_view_corr_{key.replace(" ", "_").lower()}.csv'
    res['corr'].to_csv(ANALYSIS_DIR / fname)
    print(f"Saved: {ANALYSIS_DIR / fname}")

print(f"\nAll results exported to: {ANALYSIS_DIR}")
print("\nFiles:")
for f in sorted(ANALYSIS_DIR.iterdir()):
    if f.is_file():
        sz = f.stat().st_size / 1024
        print(f"  {f.name:<60} {sz:.1f} KB")

---
## 13. Final Feature Selection Summary

Based on all evidence, produce the final recommended feature set per view.

In [ ]:
print("=" * 80)
print("FINAL FEATURE SELECTION SUMMARY")
print("=" * 80)

final_selection = {}

for name, report in redundancy_reports.items():
    # Strategy: keep features flagged by <= 3 methods
    # From multi-member clusters, keep the feature with lowest N_Flags
    # (or highest mRMR rank if available)
    keep = []
    remove = []

    cluster_res = cluster_results.get(name)
    if cluster_res:
        cluster_df = cluster_res['clusters']
        for cid in cluster_df['Cluster'].unique():
            members = cluster_df[cluster_df['Cluster'] == cid]['Feature'].tolist()
            if len(members) == 1:
                keep.append(members[0])
            else:
                # Pick the member with fewest redundancy flags
                member_flags = report[report['Feature'].isin(members)].copy()
                # Prefer features with lower N_Flags; break ties with mRMR rank
                if 'mRMR_Rank' in member_flags.columns:
                    member_flags['sort_key'] = (
                        member_flags['N_Flags'] * 100 +
                        member_flags['mRMR_Rank'].fillna(999)
                    )
                else:
                    member_flags['sort_key'] = member_flags['N_Flags']
                member_flags = member_flags.sort_values('sort_key')
                best = member_flags.iloc[0]['Feature']
                keep.append(best)
                remove.extend([m for m in members if m != best])
    else:
        # No clustering — use N_Flags threshold
        keep = report[report['N_Flags'] < 4]['Feature'].tolist()
        remove = report[report['N_Flags'] >= 4]['Feature'].tolist()

    # Also remove features with extreme VIF that aren't cluster representatives
    vif_res = vif_results.get(name)
    if vif_res is not None:
        extreme_vif = vif_res[vif_res['VIF'] > 50]['Feature'].tolist()
        for f in extreme_vif:
            if f in keep and f not in remove:
                # Only remove if another feature in same cluster is kept
                if cluster_res:
                    cdf = cluster_res['clusters']
                    f_cluster = cdf[cdf['Feature'] == f]['Cluster'].values
                    if len(f_cluster) > 0:
                        same_cluster = cdf[cdf['Cluster'] == f_cluster[0]]['Feature'].tolist()
                        others_kept = [m for m in same_cluster if m in keep and m != f]
                        if others_kept:
                            keep.remove(f)
                            remove.append(f)

    final_selection[name] = {'keep': keep, 'remove': remove}

    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")
    print(f"  Original features: {len(VIEWS[name].columns)}")
    print(f"  KEEP ({len(keep)}):   {keep}")
    print(f"  REMOVE ({len(remove)}): {remove}")
    print(f"  Reduction: {len(remove)}/{len(VIEWS[name].columns)} "
          f"({len(remove)/len(VIEWS[name].columns)*100:.0f}%) removed")

# Save final selection
import json
with open(ANALYSIS_DIR / 'final_feature_selection.json', 'w') as f:
    json.dump(final_selection, f, indent=2)
print(f"\nFinal selection saved to: {ANALYSIS_DIR / 'final_feature_selection.json'}")

In [ ]:
print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)
print(f"\nAll figures and CSV files saved to: {ANALYSIS_DIR}")
print("\nMethods applied per view (intra-view):")
print("  1. Distance Correlation (dCor) - non-linear dependencies")
print("  2. Spearman Rank Correlation - monotonic relationships")
print("  3. Pearson Correlation - linear relationships")
print("  4. Variance Inflation Factor (VIF) - multicollinearity")
print("  5. Hierarchical Clustering - feature grouping")
print("  6. mRMR - relevance vs. redundancy trade-off")
print("  7. Mutual Information - general statistical dependence")
print("  8. Condition Number - numerical stability")
print("\nCross-view analysis:")
print("  9. Cross-View Spearman & Pearson - inter-view dependencies")
print("\nViews analyzed:")
for name, df in VIEWS.items():
    print(f"  - {name}: {df.shape[1]} features")